Installazione Librerie

In [1]:
!pip install -r '/content/drive/MyDrive/Colab Notebooks/Orchestrator/requirements.txt'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.2 MB/s eta 0:00:00


Avvio MongoDB

In [2]:
# 1. Rimuovi il file di configurazione malformato che blocca apt
!rm -f /etc/apt/sources.list.d/mongodb-org-6.0.list

# 2. Pulisci la cache di apt per sbloccare il sistema
!apt-get clean
!apt-get update > /dev/null

# 3. Importa nuovamente la chiave GPG ufficiale
!wget -qO - https://www.mongodb.org/static/pgp/server-6.0.asc | gpg --dearmor > /usr/share/keyrings/mongodb-server-6.0.gpg

# 4. Aggiungi il repository con la sintassi CORRETTA (singola parentesi quadra)
!echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-6.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/6.0 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-6.0.list

# 5. Aggiorna e installa
!apt-get update > /dev/null
!apt-get install -y mongodb-org > /dev/null

# 6. Crea la directory dei dati e avvia il demone in background
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-6.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/6.0 multiverse
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
about to fork child process, waiting until server is ready for connections.
forked process: 4315
child process started successfully, parent exiting


Check stato Mongo

In [3]:
!ps aux | grep mongod

root        4315  2.2  0.7 2575964 103388 ?      Sl   16:21   0:00 mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db
root        4488  0.0  0.0   7376  3496 ?        S    16:22   0:00 /bin/bash -c ps aux | grep mongod
root        4490  0.0  0.0   6484  2340 ?        S    16:22   0:00 grep mongod


Verifica persistenza di mongo

In [ ]:
from pymongo import MongoClient
import json

def inspect_mongodb():
    print("🔍 Ispezione del Database MongoDB in corso...\n")

    # Connessione al DB locale (quello avviato tramite il demone in background)
    try:
        client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=2000)
        client.admin.command('ping') # Testa la connessione
    except Exception as e:
        print("❌ Impossibile connettersi a MongoDB. Assicurati che il demone sia in esecuzione.")
        print(f"Errore: {e}")
        return

    db = client["medfactcheck"] # Sostituisci se il tuo StorageManager usa un nome diverso

    # 1. Quanti Claim ci sono?
    claims_count = db["claims"].count_documents({})
    print(f"📊 Totale Claims elaborati: {claims_count}")

    if claims_count == 0:
        print("\nIl database è vuoto. Sembra che non ci sia persistenza o non hai ancora eseguito il sistema.")
        return

    print("\n--- ULTIMO CLAIM SALVATO ---")
    # Prendi l'ultimo claim inserito (ordinando per data/id decrescente)
    latest_claim = db["claims"].find_one(sort=[("_id", -1)])

    if latest_claim:
        print(f"ID Claim: {latest_claim.get('claim_id', 'N/A')}")
        print(f"Testo Originale: '{latest_claim.get('original_text', 'N/A')}'")
        print(f"Status: {latest_claim.get('status', 'N/A')}")

        # Mostra la scomposizione se presente
        sub_claims = latest_claim.get('sub_claims', [])
        if sub_claims:
            print("\n🔬 Sub-Claims estratti:")
            for sc in sub_claims:
                print(f"  - {sc.get('claim', 'N/A')}")

        # Cerca i verdetti associati a questo claim
        verdicts = list(db["verdicts"].find({"claim_id": latest_claim.get('claim_id')}))
        if verdicts:
            print("\n⚖️ Verdetti Finali:")
            for v in verdicts:
                print(f"  - {v.get('claim', 'N/A')[:40]}... -> [{v.get('verdict')}] ({v.get('confidence_score', 0) * 100:.1f}%)")

        # Verifica quante evidenze sono state salvate
        evidence_count = db["evidence"].count_documents({"claim_id": latest_claim.get('claim_id')})
        print(f"\n📚 Passaggi di evidenza salvati per questo claim: {evidence_count}")

        print("\n--- JSON GREZZO DEL CLAIM ---")
        # Rimuoviamo l'ObjectId per poterlo stampare formattato
        latest_claim.pop('_id', None)
        print(json.dumps(latest_claim, indent=2, ensure_ascii=False))

if __name__ == "__main__":
    inspect_mongodb()

Generazione test

In [13]:
!python '/content/drive/MyDrive/Colab Notebooks/Orchestrator/src/test.py'

Inizializzazione ambiente di test in: /content/drive/MyDrive/Colab Notebooks/Orchestrator/src/test_image

⏳ Scaricando 5 claim da SciFact (tramite MTEB) per i test puramente testuali...
🎨 Generazione di 5 finti screenshot Social in Inglese (Test OCR)...
🔗 Mappatura dei claim medici sulle immagini cliniche locali...

🚀 GENERATORE DI TEST COMPLETATO CON SUCCESSO

✅ [1] 5 CLAIM TESTUALI PURI (Da incollare in Streamlit senza immagini):
   1. PGE 2 suppresss intestinal tumor growth by altering the expression of tumor suppressing and DNA repair genes.
   2. Autophagy declines in aged organisms.
   3. Patients in stable partnerships have a slower progression from HIV to AIDS.
   4. PTEN is a regulator for the transcriptional activity of SRF
   5. APOE4 expression in iPSC-derived neurons increases AlphaBeta production and tau phosphorylation causing GABA neuron degeneration.

✅ [2] 5 URL DA TESTARE (Per verificare il Knowledge Base Routing):
   1. https://en.wikipedia.org/wiki/Paracetamol
   2

In [35]:
import os
import subprocess
import time
import re
import sys

# Spostiamoci nella cartella corretta del progetto
os.chdir('/content/drive/MyDrive/Colab Notebooks/Orchestrator/src')

print("🧹 1. Pulizia completa dei processi orfani...")
os.system("pkill -f uvicorn")
os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(2)

print("⚙️ 2. Avvio del Server FastAPI in background...")
# Avvia Uvicorn e scrive tutto dentro 'api.log'
os.system("PYTHONUNBUFFERED=1 nohup uvicorn api:app --host 0.0.0.0 --port 8000 > api.log 2>&1 &")

print("🎨 3. Avvio della Dashboard Streamlit in background...")
# Avvia Streamlit e scrive tutto dentro 'streamlit.log'
os.system("nohup streamlit run agent/utils/Dashboard.py > streamlit.log 2>&1 &")

time.sleep(6) # Diamo tempo ai modelli di caricarsi in memoria

print("🌐 4. Generazione del Tunnel Cloudflare per la Dashboard...")
tunnel_process = subprocess.Popen(
    ['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Intercettiamo il link pubblico
for line in iter(tunnel_process.stdout.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print(f"\n🚀 DASHBOARD ONLINE: {match.group(0)}")
            print("⚠️ (Attendi 5 secondi prima di cliccare per i DNS)")
            break

print("\n" + "="*70)
print("📜 5. LOG DEL SERVER FASTAPI IN TEMPO REALE")
print("💡 LASCIA QUESTA CELLA IN ESECUZIONE per mantenere tutto attivo.")
print("="*70 + "\n")

# Tailing continuo del file api.log
try:
    with open("api.log", "r") as f:
        while True:
            line = f.readline()
            if line:
                sys.stdout.write(line)
                sys.stdout.flush()
            else:
                # Nessun nuovo log: aspetta un istante per non sovraccaricare la CPU
                time.sleep(0.5)
except FileNotFoundError:
    print("Errore: Il file api.log non è stato ancora creato. Riavvia la cella.")
except KeyboardInterrupt:
    print("\n⏹️ Esecuzione interrotta. I server rimangono in background.")

🧹 1. Pulizia completa dei processi orfani...
⚙️ 2. Avvio del Server FastAPI in background...
🎨 3. Avvio della Dashboard Streamlit in background...
🌐 4. Generazione del Tunnel Cloudflare per la Dashboard...

🚀 DASHBOARD ONLINE: https://duties-governing-amend-pdf.trycloudflare.com
⚠️ (Attendi 5 secondi prima di cliccare per i DNS)

📜 5. LOG DEL SERVER FASTAPI IN TEMPO REALE
💡 LASCIA QUESTA CELLA IN ESECUZIONE per mantenere tutto attivo.

✅ URI impostata su locale: mongodb://localhost:27017/
✅ Connessione MongoDB OK → database: 'medfactcheck'
✅ Classe StorageManager definita.
⏳ Inizializzazione globale dei modelli in corso.
Caricamento di Qwen/Qwen2.5-VL-7B-Instruct in modalità Multimodale (NF4 + SDPA)...
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this cla